# Transaction Fraud Intelligence — V2

## 1. Objective and assumptions

**Does customer history improve fraud ranking within a fixed daily review budget?**

This is a synthetic development experiment. V2 keeps the four V1 comparators, hardens the simulator, diagnoses low-and-slow misses, tests all five declared seeds, and measures six feature-family ablations. A lower score on a harder simulation is not a failure.

**Run:** open a fresh Colab CPU runtime and select **Runtime → Run all**. The notebook installs its own isolated packages, runs every stage in order and offers a timestamped V2 results ZIP. No repository clone, dataset upload, package edits or kernel restart is required for supported Python 3.11–3.13.

The main development configuration has 1,200 customers and a nominal 120-day timeline. **Days 102 onward are reserved and never generated or inspected.** Every output uses development data only. Labels, scenario/context tags, hidden customer preferences and raw entity IDs cannot enter the model feature list.

Amounts are fictional standardized INR attempts, including failed payments. Scores are uncalibrated; value capture is not prevented loss or savings.

## 2. Configuration and reproducibility

The fixed seed list is **[42, 123, 2025, 31415, 27182]**. All seeds are reported, including in smoke mode. Full runs use 600 maximum boosting iterations and two CPU threads; CI uses 360 customers and 100 iterations with the same timeline, seeds and methodology.

The large source string below is the embedded experiment engine. It keeps this notebook self-contained and is generated from the readable `src/fraud_v2.py`. Scientific packages run in fresh subprocesses so they cannot conflict with modules already loaded in Colab's notebook kernel. Both Python and package versions are printed before simulation.

Do not alter parameters or select seeds after seeing results. The final period is not an available run option.

In [ ]:
ENGINE_SOURCE = "\"\"\"Self-contained V2 experiment engine, embedded verbatim in the Colab notebook.\n\nOnly development-period events are materialized. All evaluation entry points\nenforce the reserved-period boundary before accessing labels or model inputs.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nimport hashlib\nimport heapq\nimport importlib.metadata\nfrom itertools import groupby\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport shutil\nimport warnings\nimport zipfile\n\nimport joblib\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\nVERSION = \"0.2.0\"\nSEEDS = (42, 123, 2025, 31415, 27182)\nSTART = pd.Timestamp(\"2025-01-01\", tz=\"UTC\")\nCATEGORIES = [\"groceries\", \"retail\", \"dining\", \"travel\", \"digital\", \"utilities\"]\nCOUNTRIES = [\"IN\", \"SG\", \"AE\", \"GB\", \"US\"]\nMODELS = (\"rules\", \"logistic_history\", \"catboost_current\", \"catboost_history\")\nCORE_PACKAGES = (\"numpy\", \"pandas\", \"scipy\", \"scikit-learn\", \"catboost\", \"matplotlib\", \"joblib\")\nOBSERVABLE_COLUMNS = [\n    \"transaction_id\", \"timestamp\", \"customer_id\", \"merchant_id\", \"device_id\",\n    \"amount\", \"country\", \"category\", \"status\", \"outcome_available_at\",\n]\nNUMERIC_FEATURES = [\n    \"log_amount\", \"hour_sin\", \"hour_cos\", \"prior_count\", \"prior_count_30d\",\n    \"attempts_1h\", \"attempts_24h\", \"log_attempt_value_24h\", \"log_prior_median_30d\",\n    \"amount_ratio_30d\", \"amount_z_30d\", \"history_days\", \"hours_since_previous\",\n    \"new_device\", \"new_country\", \"new_merchant\", \"known_failures_1h\",\n    \"prior_accounts_on_device\", \"hour_deviation\",\n]\nCATEGORICAL_FEATURES = [\"category\", \"country\"]\nFEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES\nCURRENT_FEATURES = [\"log_amount\", \"hour_sin\", \"hour_cos\", \"category\", \"country\"]\nFEATURE_FAMILIES = {\n    \"transaction_amount\": [\"log_amount\"],\n    \"velocity_recency\": [\"attempts_1h\", \"attempts_24h\", \"log_attempt_value_24h\", \"hours_since_previous\", \"known_failures_1h\"],\n    \"novelty\": [\"new_device\", \"new_country\", \"new_merchant\"],\n    \"historical_baselines\": [\"prior_count\", \"prior_count_30d\", \"log_prior_median_30d\", \"amount_ratio_30d\", \"amount_z_30d\", \"history_days\"],\n    \"device_network\": [\"prior_accounts_on_device\"],\n    \"circadian\": [\"hour_sin\", \"hour_cos\", \"hour_deviation\"],\n}\nLONGITUDINAL_CANDIDATES = [\n    \"count_ratio_7d_prior30\", \"count_ratio_14d_prior30\", \"count_ratio_30d_prior30\",\n    \"attempt_value_ratio_7d_prior30\", \"attempt_value_ratio_14d_prior30\", \"attempt_value_ratio_30d_prior30\",\n    \"mean_ticket_ratio_7d_prior30\", \"median_ticket_ratio_7d_prior30\",\n    \"new_merchants_7d\", \"new_merchants_14d\", \"merchant_diversity_shift\",\n    \"category_concentration_shift\", \"new_category\",\n]\nDIAGNOSTIC_FEATURES = [\n    \"amount_ratio_30d\", \"amount_z_30d\", \"hours_since_previous\",\n    \"attempts_7d\", \"attempts_14d\", \"prior_count_30d\",\n    \"attempt_value_7d\", \"attempt_value_14d\", \"attempt_value_30d\",\n    \"new_merchant\", \"prior_accounts_on_device\", \"new_country\", \"hour_deviation\",\n    \"category_concentration_7d\",\n] + LONGITUDINAL_CANDIDATES\nSEPARATION_FEATURES = [\"hour_deviation\", \"hours_since_previous\", \"new_merchant\",\n                       \"prior_accounts_on_device\", \"attempts_1h\", \"new_device\",\n                       \"new_country\", \"amount_ratio_30d\"]\nRULE_WEIGHTS = {\"unusual_amount\": 25, \"new_device_high_amount\": 25, \"attempt_burst\": 20,\n                \"known_failure_sequence\": 15, \"shared_device_new_country\": 15}\nMETRICS = [\"average_precision\", \"roc_auc\", \"review_count\", \"fraud_reviewed\", \"precision\",\n           \"fraud_recall\", \"fraud_attempt_value_capture\", \"legitimate_reviewed\", \"false_positive_rate\"]\n\n\n@dataclass(frozen=True)\nclass Config:\n    customers: int = 1200\n    days: int = 120\n    merchants: int = 240\n    label_delay_days: int = 7\n    fraud_customer_fraction: float = 0.35\n    review_budgets: tuple = (20, 50, 100)\n    main_review_budget: int = 50\n    boosting_iterations: int = 600\n    threads: int = 2\n    mode: str = \"development\"\n\n    def __post_init__(self):\n        if self.days != 120 or tuple(self.review_budgets) != (20, 50, 100):\n            raise ValueError(\"V2 fixes the timeline and review budgets; change the protocol before experimentation.\")\n        if self.customers < 100 or self.merchants < 6:\n            raise ValueError(\"The declared experiment requires at least 100 customers and six merchants.\")\n\n    @property\n    def fit_at(self):\n        return START + pd.Timedelta(days=int(self.days * 0.60))\n\n    @property\n    def validation_at(self):\n        return START + pd.Timedelta(days=int(self.days * 0.72))\n\n    @property\n    def test_at(self):\n        return START + pd.Timedelta(days=int(self.days * 0.85))\n\n\ndef guard_development(frame, cfg):\n    \"\"\"Check timestamp metadata before any access to labels or predictors.\"\"\"\n    if \"timestamp\" not in frame:\n        raise ValueError(\"Timestamp metadata is required to enforce the final-test lock.\")\n    times = frame[\"timestamp\"]\n    if times.isna().any() or (times < START).any() or (times >= cfg.test_at).any():\n        raise ValueError(\"Reserved final-period or invalid timestamps are forbidden in V2.\")\n\n\ndef guard_validation(frame, cfg):\n    guard_development(frame, cfg)\n    if (frame.timestamp < cfg.validation_at).any():\n        raise ValueError(\"Evaluation accepts development-validation rows only.\")\n\n\ndef simulate(cfg, seed, hardened=True):\n    \"\"\"Generate only days [0,102); the final period has no events or labels here.\n\n    Normal and fraud RNG streams are separate so the V1-like reference preserves\n    the normal world while varying the explicitly documented fraud mechanisms.\n    \"\"\"\n    normal_seq, fraud_seq = np.random.SeedSequence(int(seed)).spawn(2)\n    rng, attack_rng = np.random.default_rng(normal_seq), np.random.default_rng(fraud_seq)\n    dev_days = int((cfg.test_at - START).days)\n    horizon = dev_days * 86400\n    merchant_ids = np.array([\"M%04d\" % i for i in range(cfg.merchants)])\n    merchants = pd.DataFrame({\"merchant_id\": merchant_ids,\n        \"category\": [CATEGORIES[i % len(CATEGORIES)] for i in range(cfg.merchants)]})\n    merchant_category = dict(zip(merchants.merchant_id, merchants.category))\n    profiles, customer_rows, events = [], [], []\n    normal_histories = defaultdict(list)\n\n    def add(p, sec, amount, device, country, merchant, fraud, scenario, context, fail_p, generator):\n        # Filter before status/label creation, not after reading a test dataframe.\n        sec = int(sec)\n        if sec < 0 or sec >= horizon:\n            return\n        row = {\"customer_id\": p[\"customer_id\"], \"second\": sec,\n               \"amount\": round(float(np.clip(amount, 1, 200000)), 2),\n               \"device_id\": str(device), \"country\": str(country),\n               \"merchant_id\": str(merchant), \"category\": merchant_category[str(merchant)],\n               \"status\": \"failed\" if generator.random() < fail_p else \"succeeded\",\n               \"delay\": int(generator.integers(2, 121)), \"is_fraud\": int(fraud),\n               \"fraud_scenario\": scenario, \"legitimate_context\": context}\n        events.append(row)\n        if not fraud:\n            normal_histories[p[\"customer_id\"]].append(row)\n\n    for i in range(cfg.customers):\n        cid = \"C%05d\" % i\n        p = {\"customer_id\": cid,\n             \"base\": float(np.clip(rng.lognormal(np.log(850), 0.8), 120, 7000)),\n             \"spread\": float(rng.uniform(0.3, 0.85)),\n             \"rate\": float(np.clip(rng.gamma(2, 0.25), 0.12, 1.3)),\n             \"hour\": int(rng.integers(7, 24)),\n             \"home\": str(rng.choice(COUNTRIES, p=[0.8, 0.06, 0.06, 0.04, 0.04])),\n             \"favourites\": rng.choice(merchant_ids, 6, replace=False),\n             \"device0\": \"phone_\" + cid, \"device1\": \"replacement_\" + cid}\n        p[\"phone_day\"] = int(rng.integers(25, dev_days - 10)) if rng.random() < 0.30 else dev_days + 1\n        travel_day = int(rng.integers(15, dev_days - 8)) if rng.random() < 0.25 else dev_days + 1\n        travel_country = str(rng.choice([c for c in COUNTRIES if c != p[\"home\"]]))\n        profiles.append(p)\n        customer_rows.append({\"customer_id\": cid, \"simulator_base_amount\": p[\"base\"],\n                              \"simulator_daily_rate\": p[\"rate\"], \"simulator_hour\": p[\"hour\"]})\n        n = int(rng.poisson(p[\"rate\"] * dev_days))\n        days = rng.integers(0, dev_days, n)\n        hours = np.mod(rng.normal(p[\"hour\"], 3, n), 24)\n        for day, hour in zip(days, hours):\n            context, country = \"routine\", p[\"home\"]\n            device = p[\"device1\"] if day >= p[\"phone_day\"] else p[\"device0\"]\n            if p[\"phone_day\"] <= day < p[\"phone_day\"] + 5:\n                context = \"new_phone\"\n            if travel_day <= day < travel_day + 6:\n                context, country = \"travel\", travel_country\n            if rng.random() < 0.025:\n                device = \"household_%04d\" % (i // 4)\n            amount = p[\"base\"] * rng.lognormal(0, p[\"spread\"])\n            if rng.random() < 0.018:\n                amount *= rng.uniform(2, 7)\n                context = \"large_purchase\"\n            merchant = rng.choice(p[\"favourites\"] if rng.random() < 0.85 else merchant_ids)\n            add(p, day * 86400 + int(hour * 3600), amount, device, country,\n                merchant, 0, \"legitimate\", context, 0.05, rng)\n        if rng.random() < 0.35:\n            day = int(rng.integers(10, dev_days))\n            for j in range(int(rng.integers(4, 11))):\n                add(p, day * 86400 + p[\"hour\"] * 3600 + j * 70,\n                    p[\"base\"] * rng.uniform(0.08, 2.2),\n                    p[\"device1\"] if day >= p[\"phone_day\"] else p[\"device0\"],\n                    p[\"home\"], rng.choice(p[\"favourites\"]), 0, \"legitimate\", \"legitimate_burst\", 0.25, rng)\n\n    selected = attack_rng.choice(len(profiles), max(3, int(cfg.customers * cfg.fraud_customer_fraction)), replace=False)\n    for k, index in enumerate(selected):\n        p = profiles[int(index)]\n        scenario = (\"account_takeover\", \"card_testing\", \"low_and_slow\")[k % 3]\n        day = int(attack_rng.integers(12, dev_days))\n        mimic = hardened and attack_rng.random() < 0.80\n        hour = float(np.mod(attack_rng.normal(p[\"hour\"], 3), 24)) if mimic else float(attack_rng.uniform(0, 24))\n        base_sec = day * 86400 + int(hour * 3600)\n        previous = [r for r in normal_histories[p[\"customer_id\"]] if r[\"second\"] < base_sec]\n        latest = max(previous, key=lambda r: r[\"second\"]) if previous else None\n        familiar_device = latest[\"device_id\"] if latest else p[\"device0\"]\n        device = familiar_device if attack_rng.random() < (0.85 if hardened else 0.45) else \"shared_actor_%03d\" % int(attack_rng.integers(0, 30))\n        country = p[\"home\"] if attack_rng.random() < (0.92 if hardened else 0.75) else str(attack_rng.choice(COUNTRIES))\n        known_merchants = [r[\"merchant_id\"] for r in previous] or list(p[\"favourites\"])\n        if scenario == \"account_takeover\":\n            count, step = int(attack_rng.integers(2, 7)), int(attack_rng.integers(120, 1800))\n            if mimic:\n                step = int(attack_rng.integers(1800, 36000))\n            fail_p = 0.05 if mimic else 0.18\n        elif scenario == \"card_testing\":\n            count, step = int(attack_rng.integers(5, 13)), int(attack_rng.integers(15, 150))\n            if hardened and attack_rng.random() < 0.50:\n                step = int(attack_rng.integers(600, 5400))\n            fail_p = float(attack_rng.uniform(0.02, 0.12)) if mimic else 0.45\n        else:\n            count, step, fail_p = int(attack_rng.integers(3, 9)), int(attack_rng.integers(1, 4)) * 86400, 0.06\n        for j in range(count):\n            event_sec = base_sec + j * step\n            if hardened and scenario == \"low_and_slow\":\n                # Per-event jitter prevents exact day/hour spacing being a tag.\n                event_sec = max(base_sec, event_sec + int(attack_rng.normal(0, 4 * 3600)))\n            if mimic and scenario != \"card_testing\":\n                amount = p[\"base\"] * attack_rng.lognormal(0, p[\"spread\"])\n            elif scenario == \"account_takeover\":\n                amount = p[\"base\"] * attack_rng.uniform(1.2, 6)\n            elif scenario == \"card_testing\":\n                amount = p[\"base\"] * (attack_rng.lognormal(-0.2, p[\"spread\"]) if mimic and attack_rng.random() < 0.55 else attack_rng.uniform(0.02, 0.35))\n            else:\n                amount = p[\"base\"] * attack_rng.lognormal(0, p[\"spread\"] * 0.7)\n            merchant = attack_rng.choice(known_merchants if attack_rng.random() < (0.80 if hardened else 0.35) else merchant_ids)\n            add(p, event_sec, amount, device, country, merchant, 1, scenario, \"not_applicable\", fail_p, attack_rng)\n    tx = pd.DataFrame(events)\n    tx[\"timestamp\"] = START + pd.to_timedelta(tx.pop(\"second\"), unit=\"s\")\n    tx[\"outcome_available_at\"] = tx.timestamp + pd.to_timedelta(tx.pop(\"delay\"), unit=\"s\")\n    tx[\"label_available_at\"] = tx.timestamp + pd.Timedelta(days=cfg.label_delay_days)\n    tx = tx.sort_values(\"timestamp\", kind=\"stable\").reset_index(drop=True)\n    tx.insert(0, \"transaction_id\", [\"T%08d\" % i for i in range(len(tx))])\n    guard_development(tx, cfg)\n    assert tx.transaction_id.is_unique and tx.timestamp.is_monotonic_increasing\n    assert (tx.amount > 0).all()\n    assert (tx.outcome_available_at > tx.timestamp).all()\n    return pd.DataFrame(customer_rows), merchants, tx\n\n\ndef concentration(rows):\n    counts = Counter(r[3] for r in rows)\n    return sum((n / len(rows)) ** 2 for n in counts.values()) if rows else np.nan\n\n\ndef build_features(tx, cfg):\n    guard_development(tx, cfg)\n    events = tx[OBSERVABLE_COLUMNS].sort_values([\"timestamp\", \"transaction_id\"])\n    state, device_accounts, pending = {}, defaultdict(set), []\n    known_failures, records = defaultdict(deque), []\n    hour_ns, day_ns = 3600 * 10**9, 86400 * 10**9\n    for timestamp, batch_iter in groupby(events.itertuples(index=False), key=lambda r: r.timestamp):\n        batch, t = list(batch_iter), int(timestamp.value)\n        while pending and pending[0][0] <= t:\n            available, event_id, cid = heapq.heappop(pending)\n            known_failures[cid].append(available)\n        flags = {}\n        for r in batch:\n            cid = r.customer_id\n            if cid not in state:\n                state[cid] = {\"history\": deque(), \"count\": 0, \"devices\": set(),\n                              \"countries\": set(), \"merchants\": set(), \"categories\": set(),\n                              \"first\": t, \"last\": None, \"sin_sum\": 0., \"cos_sum\": 0.}\n            s = state[cid]\n            while s[\"history\"] and s[\"history\"][0][0] < t - 60 * day_ns:\n                s[\"history\"].popleft()\n            h = list(s[\"history\"])\n            h30 = [x for x in h if x[0] >= t - 30 * day_ns]\n            amounts = np.array([x[1] for x in h30], dtype=float)\n            median = float(np.median(amounts)) if len(amounts) else np.nan\n            h24 = [x for x in h30 if x[0] >= t - day_ns]\n            f = known_failures[cid]\n            while f and f[0] < t - hour_ns:\n                f.popleft()\n            angle = 2 * np.pi * (timestamp.hour + timestamp.minute / 60) / 24\n            sine, cosine = float(np.sin(angle)), float(np.cos(angle))\n            norm, hour_dev = math.hypot(s[\"sin_sum\"], s[\"cos_sum\"]), np.nan\n            if s[\"count\"] >= 3 and norm > 1e-8:\n                hour_dev = float(np.arccos(np.clip((sine*s[\"sin_sum\"]+cosine*s[\"cos_sum\"])/norm, -1, 1))*12/np.pi)\n            history_days = (t - s[\"first\"]) / day_ns\n            flags[r.transaction_id] = int(r.merchant_id not in s[\"merchants\"])\n            row = {\n                \"transaction_id\": r.transaction_id, \"log_amount\": float(np.log1p(r.amount)),\n                \"hour_sin\": sine, \"hour_cos\": cosine, \"prior_count\": s[\"count\"],\n                \"prior_count_30d\": len(h30), \"attempts_1h\": sum(x[0] >= t-hour_ns for x in h30),\n                \"attempts_24h\": len(h24), \"log_attempt_value_24h\": float(np.log1p(sum(x[1] for x in h24))),\n                \"log_prior_median_30d\": float(np.log1p(median)) if len(h30) else np.nan,\n                \"amount_ratio_30d\": float(r.amount/median) if len(h30) >= 3 else np.nan,\n                \"amount_z_30d\": float((r.amount-amounts.mean())/(amounts.std()+50)) if len(h30) >= 3 else np.nan,\n                \"history_days\": history_days,\n                \"hours_since_previous\": (t-s[\"last\"])/hour_ns if s[\"last\"] is not None else np.nan,\n                \"new_device\": int(r.device_id not in s[\"devices\"]),\n                \"new_country\": int(r.country not in s[\"countries\"]),\n                \"new_merchant\": flags[r.transaction_id], \"known_failures_1h\": len(f),\n                \"prior_accounts_on_device\": len(device_accounts[r.device_id]), \"hour_deviation\": hour_dev,\n                \"category\": str(r.category), \"country\": str(r.country),\n                \"new_category\": int(r.category not in s[\"categories\"]),\n                \"attempt_value_30d\": float(sum(x[1] for x in h30)),\n            }\n            for days in (7, 14, 30):\n                recent = [x for x in h if x[0] >= t-days*day_ns]\n                # Disjoint baseline [t-(days+30), t-days); no overlapping denominator.\n                baseline = [x for x in h if t-(days+30)*day_ns <= x[0] < t-days*day_ns]\n                rv, bv = [x[1] for x in recent], [x[1] for x in baseline]\n                reliable = len(baseline) >= 3 and history_days >= days + 30\n                row[\"attempts_%dd\" % days] = len(recent)\n                row[\"attempt_value_%dd\" % days] = float(sum(rv))\n                row[\"count_ratio_%dd_prior30\" % days] = len(recent)/(len(baseline)*days/30) if reliable else np.nan\n                row[\"attempt_value_ratio_%dd_prior30\" % days] = sum(rv)/(sum(bv)*days/30) if reliable else np.nan\n                row[\"new_merchants_%dd\" % days] = len({x[2] for x in recent if x[4]})\n                if days == 7:\n                    enough = reliable and len(recent) >= 3\n                    row[\"mean_ticket_ratio_7d_prior30\"] = float(np.mean(rv)/np.mean(bv)) if enough else np.nan\n                    row[\"median_ticket_ratio_7d_prior30\"] = float(np.median(rv)/np.median(bv)) if enough else np.nan\n                    row[\"merchant_diversity_shift\"] = (len({x[2] for x in recent})/len(recent)-len({x[2] for x in baseline})/len(baseline)) if enough else np.nan\n                    row[\"category_concentration_7d\"] = concentration(recent)\n                    row[\"category_concentration_shift\"] = concentration(recent)-concentration(baseline) if enough else np.nan\n            records.append(row)\n        # No state updates until every transaction at this timestamp has been scored.\n        for r in batch:\n            s = state[r.customer_id]\n            angle = 2*np.pi*(timestamp.hour+timestamp.minute/60)/24\n            s[\"history\"].append((t, float(r.amount), r.merchant_id, r.category, flags[r.transaction_id]))\n            s[\"count\"] += 1\n            s[\"last\"] = t\n            for key, value in [(\"devices\", r.device_id), (\"countries\", r.country),\n                               (\"merchants\", r.merchant_id), (\"categories\", r.category)]:\n                s[key].add(value)\n            s[\"sin_sum\"] += float(np.sin(angle))\n            s[\"cos_sum\"] += float(np.cos(angle))\n            device_accounts[r.device_id].add(r.customer_id)\n            if r.status == \"failed\":\n                heapq.heappush(pending, (int(r.outcome_available_at.value), r.transaction_id, r.customer_id))\n    result = pd.DataFrame(records).set_index(\"transaction_id\")\n    numeric = result.select_dtypes(include=\"number\")\n    if np.isinf(numeric.to_numpy()).any():\n        raise ValueError(\"Non-finite historical feature.\")\n    return result\n\n\ndef temporal_split(tx, cfg):\n    guard_development(tx, cfg)\n    masks = {\n        \"train\": (tx.timestamp < cfg.fit_at) & (tx.label_available_at <= cfg.fit_at),\n        \"early_stop\": (tx.timestamp >= cfg.fit_at) & (tx.timestamp < cfg.validation_at)\n                      & (tx.label_available_at <= cfg.validation_at),\n        \"validation\": tx.timestamp >= cfg.validation_at,\n    }\n    audit = []\n    for name, mask in masks.items():\n        part = tx.loc[mask]\n        if part.is_fraud.nunique() != 2:\n            raise ValueError(name + \" needs both classes; the declared run failed rather than skipping this seed.\")\n        audit.append({\"period\": name, \"rows\": len(part), \"frauds\": int(part.is_fraud.sum()),\n                      \"fraud_rate\": float(part.is_fraud.mean()),\n                      \"first_event\": str(part.timestamp.min()), \"last_event\": str(part.timestamp.max()),\n                      \"label_cutoff\": str(cfg.fit_at if name == \"train\" else cfg.validation_at) if name != \"validation\" else \"retrospective validation labels\"})\n    assert (sum(mask.astype(int) for mask in masks.values()) <= 1).all()\n    assert (tx.loc[masks[\"train\"], \"label_available_at\"] <= cfg.fit_at).all()\n    assert (tx.loc[masks[\"early_stop\"], \"label_available_at\"] <= cfg.validation_at).all()\n    return masks, pd.DataFrame(audit)\n\n\ndef timing_checks(tx, features, cfg):\n    times = [START, START+pd.Timedelta(seconds=30), START+pd.Timedelta(seconds=30), START+pd.Timedelta(seconds=120)]\n    probe = pd.DataFrame({\"transaction_id\": [\"p0\", \"p1\", \"p2\", \"p3\"], \"timestamp\": times,\n        \"customer_id\": [\"c\"]*4, \"merchant_id\": [\"m\"]*4, \"device_id\": [\"d\"]*4,\n        \"amount\": [100., 200., 400., 800.], \"country\": [\"IN\"]*4, \"category\": [\"retail\"]*4,\n        \"status\": [\"failed\", \"succeeded\", \"succeeded\", \"succeeded\"],\n        \"outcome_available_at\": [t+pd.Timedelta(seconds=60) for t in times]})\n    f = build_features(probe, cfg)\n    assert f.loc[\"p1\", \"prior_count\"] == f.loc[\"p2\", \"prior_count\"] == 1\n    assert f.loc[\"p1\", \"known_failures_1h\"] == 0 and f.loc[\"p3\", \"known_failures_1h\"] == 1\n    assert np.isclose(np.expm1(f.loc[\"p1\", \"log_prior_median_30d\"]), 100)\n    assert np.isclose(f.loc[\"p3\", \"amount_ratio_30d\"], 4.)\n    before = build_features(probe.iloc[:3], cfg)\n    pd.testing.assert_frame_equal(before, f.loc[before.index])\n    # Test the real development prefix, including every new longitudinal measure.\n    boundary = tx.timestamp.iloc[min(1200, len(tx)-1)]\n    prefix = tx.loc[tx.timestamp <= boundary]\n    rebuilt = build_features(prefix, cfg)\n    pd.testing.assert_frame_equal(rebuilt, features.loc[rebuilt.index])\n    return {name: \"passed\" for name in [\"future_invariance\", \"same_timestamp_batching\",\n            \"delayed_outcomes\", \"current_amount_excluded\", \"label_maturity\", \"development_boundary\"]}\n\n\ndef prepare_seed(cfg, seed):\n    _, _, tx = simulate(cfg, seed)\n    f = build_features(tx, cfg)\n    tx = tx.set_index(\"transaction_id\")\n    assert f.index.equals(tx.index)\n    masks, audit = temporal_split(tx, cfg)\n    checks = timing_checks(tx.reset_index(), f, cfg)\n    return {\"seed\": int(seed), \"tx\": tx, \"features\": f, \"masks\": masks, \"split_audit\": audit,\n            \"checks\": checks, \"models\": {}, \"predictions\": {}, \"model_columns\": {}}\n\n\ndef rule_triggers(frame):\n    established = frame.prior_count >= 5\n    return pd.DataFrame({\n        \"unusual_amount\": established & (frame.amount_ratio_30d >= 4),\n        \"new_device_high_amount\": established & frame.new_device.eq(1) & (frame.amount_ratio_30d >= 2),\n        \"attempt_burst\": frame.attempts_1h >= 4,\n        \"known_failure_sequence\": frame.known_failures_1h >= 2,\n        \"shared_device_new_country\": established & (frame.prior_accounts_on_device >= 3) & frame.new_country.eq(1),\n    }, index=frame.index).astype(int)\n\n\ndef rule_scores(frame):\n    return rule_triggers(frame).mul(pd.Series(RULE_WEIGHTS)).sum(axis=1).to_numpy(dtype=float)\n\n\ndef fit_detector(state, cfg, name, columns=None):\n    guard_development(state[\"tx\"], cfg)\n    f, y, masks = state[\"features\"], state[\"tx\"].is_fraud.astype(int), state[\"masks\"]\n    columns = list(columns or (CURRENT_FEATURES if name == \"catboost_current\" else FEATURES))\n    forbidden = {\"is_fraud\", \"fraud_scenario\", \"legitimate_context\", \"customer_id\", \"device_id\", \"status\"}\n    if set(columns) & forbidden or not set(columns) <= set(FEATURES + LONGITUDINAL_CANDIDATES):\n        raise ValueError(\"Undeclared model feature; simulator-only columns are forbidden.\")\n    if name == \"logistic_history\":\n        pre = ColumnTransformer([\n            (\"numeric\", Pipeline([(\"impute\", SimpleImputer(strategy=\"median\", add_indicator=True, keep_empty_features=True)),\n                                  (\"scale\", StandardScaler())]), NUMERIC_FEATURES),\n            (\"categorical\", OneHotEncoder(handle_unknown=\"ignore\", sparse_output=True), CATEGORICAL_FEATURES)])\n        model = Pipeline([(\"preprocess\", pre), (\"classifier\", LogisticRegression(\n            C=1., max_iter=1500, solver=\"lbfgs\", random_state=state[\"seed\"]))])\n        with warnings.catch_warnings(record=True) as caught:\n            warnings.simplefilter(\"always\", ConvergenceWarning)\n            model.fit(f.loc[masks[\"train\"], columns], y.loc[masks[\"train\"]])\n            if any(issubclass(w.category, ConvergenceWarning) for w in caught):\n                raise RuntimeError(\"Logistic regression did not converge.\")\n    else:\n        model = CatBoostClassifier(iterations=cfg.boosting_iterations, depth=6, learning_rate=0.06,\n            l2_leaf_reg=5, loss_function=\"Logloss\", eval_metric=\"Logloss\", random_seed=state[\"seed\"],\n            thread_count=cfg.threads, allow_writing_files=False)\n        model.fit(f.loc[masks[\"train\"], columns], y.loc[masks[\"train\"]],\n            cat_features=[c for c in CATEGORICAL_FEATURES if c in columns],\n            eval_set=(f.loc[masks[\"early_stop\"], columns], y.loc[masks[\"early_stop\"]]),\n            early_stopping_rounds=50, verbose=False)\n    return model, columns\n\n\ndef ratio(a, b):\n    return float(a / b) if b else np.nan\n\n\ndef select_daily(frame, k):\n    ordered = frame.reset_index().sort_values([\"day\", \"score\", \"transaction_id\"], ascending=[True, False, True])\n    chosen = ordered.groupby(\"day\", sort=False).head(k).transaction_id\n    return pd.Series(frame.index.isin(chosen), index=frame.index)\n\n\ndef evaluate_scores(tx, scores, name, cfg):\n    guard_validation(tx, cfg)\n    frame = tx[[\"timestamp\", \"customer_id\", \"amount\", \"is_fraud\", \"fraud_scenario\",\n                \"legitimate_context\", \"category\", \"country\"]].copy()\n    frame[\"score\"], frame[\"model\"] = np.asarray(scores), name\n    if not np.isfinite(frame.score).all():\n        raise ValueError(\"Non-finite validation scores.\")\n    frame[\"day\"] = frame.timestamp.dt.floor(\"D\")\n    fraud_n, legit_n = int(frame.is_fraud.sum()), int(frame.is_fraud.eq(0).sum())\n    fraud_value = float(frame.loc[frame.is_fraud.eq(1), \"amount\"].sum())\n    ap = float(average_precision_score(frame.is_fraud, frame.score))\n    auc = float(roc_auc_score(frame.is_fraud, frame.score))\n    rows, daily = [], []\n    calendar = pd.date_range(cfg.validation_at, cfg.test_at, inclusive=\"left\", freq=\"D\")\n    for k in cfg.review_budgets:\n        selected = select_daily(frame, k)\n        picked, hits = frame.loc[selected], frame.loc[selected & frame.is_fraud.eq(1)]\n        false_n = int(picked.is_fraud.eq(0).sum())\n        rows.append({\"model\": name, \"daily_review_cap\": k, \"average_precision\": ap,\n            \"roc_auc\": auc, \"review_count\": len(picked), \"fraud_reviewed\": len(hits),\n            \"precision\": ratio(len(hits), len(picked)), \"fraud_recall\": ratio(len(hits), fraud_n),\n            \"fraud_attempt_value_capture\": ratio(float(hits.amount.sum()), fraud_value),\n            \"legitimate_reviewed\": false_n, \"false_positive_rate\": ratio(false_n, legit_n)})\n        for day in calendar:\n            group = frame.loc[frame.day.eq(day)]\n            reviewed = group.loc[selected.loc[group.index]]\n            hit = reviewed.loc[reviewed.is_fraud.eq(1)]\n            daily.append({\"model\": name, \"daily_review_cap\": k, \"day\": str(day),\n                \"transactions\": len(group), \"frauds\": int(group.is_fraud.sum()),\n                \"reviews\": len(reviewed), \"fraud_reviewed\": len(hit),\n                \"legitimate_reviewed\": int(reviewed.is_fraud.eq(0).sum()),\n                \"precision\": ratio(len(hit), len(reviewed)), \"recall\": ratio(len(hit), int(group.is_fraud.sum())),\n                \"false_positive_rate\": ratio(int(reviewed.is_fraud.eq(0).sum()), int(group.is_fraud.eq(0).sum())),\n                \"fraud_attempt_value_capture\": ratio(float(hit.amount.sum()), float(group.loc[group.is_fraud.eq(1), \"amount\"].sum())),\n                \"budget_exhausted\": len(reviewed) == k, \"fewer_candidates_than_budget\": len(group) < k,\n                \"unused_capacity\": max(0, k-len(reviewed))})\n        if k == cfg.main_review_budget:\n            frame[\"selected_for_review\"] = selected\n            frame[\"action\"] = np.where(selected, \"REVIEW\", \"NO_REVIEW\")\n    return pd.DataFrame(rows), pd.DataFrame(daily), frame\n\n\ndef score_detector(state, cfg, name, model=None, columns=None):\n    mask = state[\"masks\"][\"validation\"]\n    tx = state[\"tx\"].loc[mask]\n    guard_validation(tx, cfg)\n    f = state[\"features\"].loc[mask]\n    scores = rule_scores(f) if name == \"rules\" else model.predict_proba(f[columns])[:, 1]\n    return evaluate_scores(tx, scores, name, cfg)\n\n\ndef complete_primary(state, cfg):\n    metrics, daily = [], []\n    for name in MODELS:\n        if name != \"rules\" and name not in state[\"models\"]:\n            state[\"models\"][name], state[\"model_columns\"][name] = fit_detector(state, cfg, name)\n        result = score_detector(state, cfg, name, state[\"models\"].get(name), state[\"model_columns\"].get(name))\n        m, d, p = result\n        metrics.append(m)\n        daily.append(d)\n        state[\"predictions\"][name] = p\n    state[\"comparison\"], state[\"daily_metrics\"] = pd.concat(metrics, ignore_index=True), pd.concat(daily, ignore_index=True)\n    op = state[\"comparison\"].loc[lambda x: x.daily_review_cap.eq(cfg.main_review_budget)]\n    state[\"leader\"] = op.sort_values([\"precision\", \"fraud_attempt_value_capture\", \"average_precision\", \"model\"],\n        ascending=[False, False, False, True]).iloc[0][\"model\"]\n    return state\n\n\ndef scenario_table(predictions):\n    rows = []\n    for name, frame in predictions.items():\n        for scenario, group in frame.loc[frame.is_fraud.eq(1)].groupby(\"fraud_scenario\"):\n            picked = group.loc[group.selected_for_review]\n            rows.append({\"model\": name, \"scenario\": scenario, \"fraud_count\": len(group),\n                         \"fraud_reviewed\": len(picked), \"recall\": ratio(len(picked), len(group)),\n                         \"attempt_value_capture\": ratio(float(picked.amount.sum()), float(group.amount.sum()))})\n    return pd.DataFrame(rows)\n\n\ndef evidence_tags(row):\n    tags = []\n    for condition, label in [\n        (row.prior_count < 5, \"limited_history\"), (row.amount_ratio_30d >= 2, \"amount_at_least_2x_history\"),\n        (row.new_device == 1, \"unseen_device\"), (row.new_country == 1, \"unseen_country\"),\n        (row.new_merchant == 1, \"unseen_merchant\"), (row.attempts_1h >= 4, \"recent_attempt_burst\"),\n        (row.known_failures_1h >= 2, \"known_failure_sequence\"),\n        (row.prior_accounts_on_device >= 3, \"device_previously_used_by_3plus_accounts\"),\n        (row.hour_deviation >= 6, \"hour_deviation_at_least_6h\")]:\n        if condition:\n            tags.append(label)\n    return tags\n\n\ndef evidence_text(row):\n    parts = [\"%d earlier attempts in 1 hour\" % row.attempts_1h,\n             \"%d failed outcomes known in 1 hour\" % row.known_failures_1h,\n             \"%d previously observed accounts on device\" % row.prior_accounts_on_device]\n    if pd.notna(row.amount_ratio_30d):\n        parts.insert(0, \"amount %.2fx prior 30-day median attempted amount\" % row.amount_ratio_30d)\n    return \"; \".join(parts + evidence_tags(row))\n\n\ndef enrich_predictions(state):\n    f = state[\"features\"].loc[state[\"masks\"][\"validation\"]]\n    rules = rule_triggers(f).apply(lambda row: \"; \".join(n for n in RULE_WEIGHTS if row[n]), axis=1)\n    text = f.apply(evidence_text, axis=1)\n    patterns = f.apply(lambda row: \"; \".join(evidence_tags(row)), axis=1)\n    for p in state[\"predictions\"].values():\n        p[\"triggered_rules\"], p[\"observed_evidence\"], p[\"evidence_patterns\"] = rules, text, patterns\n\n\ndef context_table(state):\n    enrich_predictions(state)\n    rows, examples = [], []\n    contexts = (\"routine\", \"travel\", \"new_phone\", \"large_purchase\", \"legitimate_burst\")\n    for name, frame in state[\"predictions\"].items():\n        for context in contexts:\n            group = frame.loc[frame.is_fraud.eq(0) & frame.legitimate_context.eq(context)]\n            picked = group.loc[group.selected_for_review]\n            patterns = Counter(tag for value in picked.evidence_patterns for tag in value.split(\"; \") if tag)\n            rows.append({\"model\": name, \"context\": context, \"legitimate_count\": len(group),\n                \"legitimate_reviewed\": len(picked), \"alert_rate\": ratio(len(picked), len(group)),\n                \"score_median\": float(group.score.median()), \"score_q75\": float(group.score.quantile(.75)),\n                \"reviewed_score_median\": float(picked.score.median()), \"reviewed_score_q75\": float(picked.score.quantile(.75)),\n                \"top_evidence_patterns\": json.dumps(patterns.most_common(3)),\n                \"provisional_leader\": name == state[\"leader\"]})\n            if name == state[\"leader\"] and len(picked):\n                # At most three distinct representative false positives per context.\n                ids = [picked.score.idxmax(), (picked.score-picked.score.median()).abs().idxmin(), picked.amount.idxmax()]\n                example = picked.loc[list(dict.fromkeys(ids))].reset_index()\n                examples.extend(example.to_dict(\"records\"))\n    return pd.DataFrame(rows), pd.DataFrame(examples, columns=[\"transaction_id\"] + list(next(iter(state[\"predictions\"].values())).columns))\n\n\ndef separation_audit(tx, features, cfg, world):\n    guard_validation(tx, cfg)\n    rows = []\n    for col in SEPARATION_FEATURES:\n        values = features.loc[tx.index, col]\n        finite = values.notna()\n        y = tx.is_fraud\n        auc = float(roc_auc_score(y.loc[finite], values.loc[finite])) if y.loc[finite].nunique() == 2 else np.nan\n        missing_auc = float(roc_auc_score(y, (~finite).astype(int))) if y.nunique() == 2 else np.nan\n        for label, name in [(0, \"legitimate\"), (1, \"fraud\")]:\n            group = values.loc[y.eq(label)]\n            rows.append({\"world\": world, \"feature\": col, \"class\": name, \"rows\": len(group),\n                \"finite_count\": int(group.notna().sum()), \"missing_rate\": float(group.isna().mean()),\n                \"mean\": float(group.mean()), \"std\": float(group.std()), \"q25\": float(group.quantile(.25)),\n                \"median\": float(group.median()), \"q75\": float(group.quantile(.75)), \"q95\": float(group.quantile(.95)),\n                \"roc_auc_raw_direction\": auc, \"roc_auc_best_direction\": max(auc, 1-auc) if np.isfinite(auc) else np.nan,\n                \"missingness_roc_auc\": missing_auc})\n    return pd.DataFrame(rows)\n\n\ndef overlap_audit(tx, features, cfg, world):\n    guard_validation(tx, cfg)\n    f = features.loc[tx.index]\n    conditions = {\"familiar_device\": f.new_device.eq(0), \"familiar_country\": f.new_country.eq(0),\n        \"familiar_merchant\": f.new_merchant.eq(0), \"normal_amount_0.5_to_2x\": f.amount_ratio_30d.between(.5, 2),\n        \"hour_deviation_under_3h\": f.hour_deviation.le(3), \"no_known_failed_outcome_1h\": f.known_failures_1h.eq(0),\n        \"fewer_than_4_prior_attempts_1h\": f.attempts_1h.lt(4)}\n    conditions[\"all_familiar_normal_conditions\"] = pd.concat(conditions, axis=1).all(axis=1)\n    rows = []\n    for label, name in [(0, \"legitimate\"), (1, \"fraud\")]:\n        mask = tx.is_fraud.eq(label)\n        for condition, passed in conditions.items():\n            rows.append({\"world\": world, \"class\": name, \"condition\": condition,\n                \"rows\": int(mask.sum()), \"count\": int(passed.loc[mask].sum()),\n                \"share\": float(passed.loc[mask].mean())})\n    return pd.DataFrame(rows)\n\n\ndef low_slow_diagnostics(state):\n    frame = state[\"predictions\"][\"catboost_history\"]\n    groups = {\n        \"caught_low_and_slow\": frame.fraud_scenario.eq(\"low_and_slow\") & frame.selected_for_review,\n        \"missed_low_and_slow\": frame.fraud_scenario.eq(\"low_and_slow\") & ~frame.selected_for_review,\n        \"legitimate_routine\": frame.is_fraud.eq(0) & frame.legitimate_context.eq(\"routine\"),\n    }\n    rows = []\n    f = state[\"features\"].loc[frame.index]\n    for group, mask in groups.items():\n        for col in DIAGNOSTIC_FEATURES:\n            v = f.loc[mask, col]\n            rows.append({\"group\": group, \"feature\": col, \"rows\": len(v), \"finite_count\": int(v.notna().sum()),\n                         \"missing_rate\": float(v.isna().mean()), \"mean\": float(v.mean()),\n                         \"std\": float(v.std()), \"q25\": float(v.quantile(.25)),\n                         \"median\": float(v.median()), \"q75\": float(v.quantile(.75))})\n    decisions = []\n    for col in LONGITUDINAL_CANDIDATES:\n        a, b = f.loc[groups[\"missed_low_and_slow\"], col], f.loc[groups[\"legitimate_routine\"], col]\n        aa, bb = a.dropna(), b.dropna()\n        denom = np.sqrt((aa.var()+bb.var())/2) if min(len(aa), len(bb)) >= 2 else np.nan\n        effect = float((aa.mean()-bb.mean())/denom) if np.isfinite(denom) and denom > 0 else np.nan\n        enough = min(len(aa), len(bb)) >= 5 and min(ratio(len(aa), len(a)), ratio(len(bb), len(b))) >= .5\n        selected = bool(enough and np.isfinite(effect) and abs(effect) >= .25)\n        decisions.append({\"feature\": col, \"diagnostic_seed\": state[\"seed\"],\n            \"missed_finite_count\": len(aa), \"routine_finite_count\": len(bb),\n            \"standardized_mean_difference\": effect, \"selected\": selected,\n            \"reason\": \"predeclared_development_diagnostic_gate_passed\" if selected else \"insufficient_coverage_or_development_separation\",\n            \"claim_status\": \"exploratory_validation_selection; no final-test evidence\"})\n    return pd.DataFrame(rows), pd.DataFrame(decisions)\n\n\ndef longitudinal_challenger(state, cfg, selected):\n    base = state[\"comparison\"].loc[lambda x: x.model.eq(\"catboost_history\") & x.daily_review_cap.eq(50)].iloc[0]\n    baseline_scenario = scenario_table({\"catboost_history\": state[\"predictions\"][\"catboost_history\"]})\n    base_low = baseline_scenario.loc[baseline_scenario.scenario.eq(\"low_and_slow\"), \"recall\"]\n    row = {\"seed\": state[\"seed\"], \"status\": \"not_run_no_features_passed_diagnostic_gate\",\n           \"selected_features\": json.dumps(selected), \"daily_review_cap\": 50,\n           \"selection_period\": \"seed_42_development_validation\",\n           \"evidence_role\": \"exploratory_selection_seed\" if state[\"seed\"] == SEEDS[0] else \"separate_synthetic_seed_replication\"}\n    for metric in METRICS:\n        row[\"baseline_\" + metric] = base[metric]\n        row[\"challenger_\" + metric] = np.nan\n        row[\"delta_\" + metric] = np.nan\n    row[\"baseline_low_and_slow_recall\"] = float(base_low.iloc[0]) if len(base_low) else np.nan\n    row[\"challenger_low_and_slow_recall\"] = row[\"delta_low_and_slow_recall\"] = np.nan\n    contexts = []\n    if selected:\n        model, columns = fit_detector(state, cfg, \"longitudinal_challenger\", FEATURES + selected)\n        metrics, _, frame = score_detector(state, cfg, \"longitudinal_challenger\", model, columns)\n        op = metrics.loc[metrics.daily_review_cap.eq(50)].iloc[0]\n        row[\"status\"] = \"evaluated_on_development_validation\"\n        for metric in METRICS:\n            row[\"challenger_\" + metric], row[\"delta_\" + metric] = op[metric], op[metric] - base[metric]\n        slow = frame.loc[frame.fraud_scenario.eq(\"low_and_slow\")]\n        row[\"challenger_low_and_slow_recall\"] = ratio(int(slow.selected_for_review.sum()), len(slow))\n        row[\"delta_low_and_slow_recall\"] = row[\"challenger_low_and_slow_recall\"] - row[\"baseline_low_and_slow_recall\"]\n        old = state[\"predictions\"][\"catboost_history\"]\n        for context in [\"routine\", \"travel\", \"new_phone\", \"large_purchase\", \"legitimate_burst\"]:\n            mask = frame.is_fraud.eq(0) & frame.legitimate_context.eq(context)\n            before, after = int(old.loc[mask].selected_for_review.sum()), int(frame.loc[mask].selected_for_review.sum())\n            contexts.append({\"seed\": state[\"seed\"], \"context\": context, \"legitimate_count\": int(mask.sum()),\n                \"baseline_legitimate_reviews\": before, \"challenger_legitimate_reviews\": after,\n                \"delta_legitimate_reviews\": after-before, \"delta_alert_rate\": ratio(after-before, int(mask.sum()))})\n        state[\"longitudinal_model\"], state[\"longitudinal_columns\"], state[\"longitudinal_predictions\"] = model, columns, frame\n    return row, contexts\n\n\ndef daily_summary(daily):\n    rows = []\n    for (model, k), g in daily.groupby([\"model\", \"daily_review_cap\"], sort=False):\n        row = {\"model\": model, \"daily_review_cap\": k, \"validation_days\": len(g),\n            \"days_budget_exhausted\": int(g.budget_exhausted.sum()),\n            \"days_fewer_candidates_than_budget\": int(g.fewer_candidates_than_budget.sum()),\n            \"days_without_candidates\": int(g.transactions.eq(0).sum()),\n            \"unused_review_slots\": int(g.unused_capacity.sum())}\n        for metric in [\"precision\", \"recall\", \"false_positive_rate\", \"fraud_attempt_value_capture\", \"reviews\"]:\n            for stat in [\"mean\", \"std\", \"min\", \"max\"]:\n                row[metric + \"_daily_\" + stat] = float(getattr(g[metric], stat)())\n            row[metric + \"_defined_days\"] = int(g[metric].notna().sum())\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef save_primary(state, cfg, out):\n    out.mkdir(parents=True, exist_ok=True)\n    guard_development(state[\"tx\"], cfg)\n    state[\"comparison\"].to_csv(out / \"comparison.csv\", index=False)\n    state[\"daily_metrics\"].to_csv(out / \"daily_metrics.csv\", index=False)\n    daily_summary(state[\"daily_metrics\"]).to_csv(out / \"daily_variability_summary.csv\", index=False)\n    scenario_table(state[\"predictions\"]).to_csv(out / \"scenario_breakdown.csv\", index=False)\n    contexts, examples = context_table(state)\n    contexts.to_csv(out / \"legitimate_context_breakdown.csv\", index=False)\n    examples.to_csv(out / \"false_positive_examples.csv\", index=False)\n    state[\"split_audit\"].to_csv(out / \"split_audit.csv\", index=False)\n    for name, frame in state[\"predictions\"].items():\n        guard_validation(frame, cfg)\n        frame.to_csv(out / (name + \"_validation_predictions.csv\"), index_label=\"transaction_id\")\n        cases = pd.concat([\n            frame.loc[frame.selected_for_review & frame.is_fraud.eq(0)].nlargest(50, \"score\").assign(case_type=\"false_positive\"),\n            frame.loc[~frame.selected_for_review & frame.is_fraud.eq(1)].nlargest(50, \"amount\").assign(case_type=\"missed_fraud\"),\n            frame.loc[frame.selected_for_review & frame.is_fraud.eq(1)].nlargest(20, \"score\").assign(case_type=\"true_positive\"),\n        ])\n        cases.to_csv(out / (name + \"_error_cases.csv\"), index_label=\"transaction_id\")\n\n\ndef validation_chart(state, cfg, out):\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n    for name, p in state[\"predictions\"].items():\n        guard_validation(p, cfg)\n        precision, recall, _ = precision_recall_curve(p.is_fraud, p.score)\n        axes[0].plot(recall, precision, label=name)\n        g = state[\"comparison\"].loc[lambda x: x.model.eq(name)]\n        axes[1].plot(g.daily_review_cap, g.fraud_recall, marker=\"o\", label=name)\n    axes[0].set(xlabel=\"Fraud recall\", ylabel=\"Precision\", title=\"Development validation: precision–recall\")\n    axes[1].set(xlabel=\"Reviews per UTC day\", ylabel=\"Fraud recall\", title=\"Same capacity for every detector\")\n    for ax in axes:\n        ax.legend(fontsize=7)\n        ax.grid(alpha=.2)\n    fig.tight_layout()\n    fig.savefig(out / \"validation_comparison.png\", dpi=150)\n    plt.close(fig)\n\n\ndef robustness_summary(per_seed):\n    raw = pd.concat(per_seed, ignore_index=True)\n    raw.insert(0, \"row_type\", \"seed\")\n    aggregates = []\n    for (model, k), group in raw.groupby([\"model\", \"daily_review_cap\"], sort=False):\n        for stat in [\"mean\", \"std\", \"min\", \"max\"]:\n            row = {\"row_type\": stat, \"seed\": np.nan, \"model\": model, \"daily_review_cap\": k}\n            row.update({metric: float(getattr(group[metric], stat)()) for metric in METRICS})\n            aggregates.append(row)\n    return pd.concat([raw, pd.DataFrame(aggregates)], ignore_index=True)\n\n\ndef robustness_chart(summary, out):\n    table = summary.loc[summary.row_type.eq(\"seed\") & summary.daily_review_cap.eq(50)].pivot(index=\"model\", columns=\"seed\", values=\"average_precision\")\n    table = table.reindex(index=MODELS, columns=SEEDS)\n    ranks = table.rank(axis=0, ascending=False, method=\"min\")\n    fig, ax = plt.subplots(figsize=(8.5, 3.5))\n    im = ax.imshow(table.to_numpy(), vmin=0, vmax=1, cmap=\"Blues\", aspect=\"auto\")\n    for i in range(len(table)):\n        for j in range(len(SEEDS)):\n            value = table.iloc[i, j]\n            ax.text(j, i, \"%.3f\\nrank %d\" % (value, ranks.iloc[i, j]), ha=\"center\", va=\"center\",\n                    color=\"white\" if value > .6 else \"black\", fontsize=9)\n    ax.set_xticks(range(len(SEEDS)), labels=[str(x) for x in SEEDS])\n    ax.set_yticks(range(len(MODELS)), labels=MODELS)\n    ax.set(title=\"Average precision and rank across all five declared seeds\", xlabel=\"Simulation seed\")\n    fig.colorbar(im, ax=ax, label=\"Average precision\")\n    fig.tight_layout()\n    fig.savefig(out / \"robustness_ranking.png\", dpi=150)\n    plt.close(fig)\n\n\ndef run_ablation(state, cfg):\n    base = state[\"comparison\"].loc[lambda x: x.model.eq(\"catboost_history\") & x.daily_review_cap.eq(50)].iloc[0].to_dict()\n    rows = [dict(base, seed=state[\"seed\"], ablation=\"full_history_reference\", removed_features=\"[]\", best_iteration=state[\"models\"][\"catboost_history\"].get_best_iteration(), **{\"delta_\"+m: 0. for m in METRICS})]\n    for family, removed in FEATURE_FAMILIES.items():\n        columns = [f for f in FEATURES if f not in removed]\n        model, columns = fit_detector(state, cfg, \"ablation_\"+family, columns)\n        metrics, _, _ = score_detector(state, cfg, \"ablation_\"+family, model, columns)\n        row = metrics.loc[metrics.daily_review_cap.eq(50)].iloc[0].to_dict()\n        row.update({\"seed\": state[\"seed\"], \"ablation\": \"without_\"+family,\n                    \"removed_features\": json.dumps(removed), \"best_iteration\": model.get_best_iteration()})\n        row.update({\"delta_\"+m: row[m]-base[m] for m in METRICS})\n        rows.append(row)\n        print(\"Ablation completed:\", family, flush=True)\n    return pd.DataFrame(rows)\n\nSTAGES = (\"simulation\", \"history\", \"split\", \"rules\", \"logistic\", \"current\", \"history_model\",\n          \"comparison\", \"scenario\", \"low_slow\", \"false_positives\", \"robustness\", \"ablation\", \"export\")\nREQUIRED_OUTPUTS = (\n    \"comparison.csv\", \"daily_metrics.csv\", \"daily_variability_summary.csv\", \"scenario_breakdown.csv\",\n    \"legitimate_context_breakdown.csv\", \"false_positive_examples.csv\", \"feature_importance.csv\",\n    \"feature_separation_audit.csv\", \"simulator_overlap_audit.csv\", \"low_and_slow_diagnostics.csv\",\n    \"longitudinal_feature_decisions.csv\", \"longitudinal_tradeoffs.csv\", \"longitudinal_context_tradeoffs.csv\",\n    \"robustness_summary.csv\", \"ablation_summary.csv\", \"split_audit.csv\", \"manifest.json\",\n    \"validation_comparison.png\", \"robustness_ranking.png\", \"feature_separation.png\",\n    \"logistic_history.joblib\", \"catboost_current.cbm\", \"catboost_history.cbm\",\n) + tuple(name+suffix for name in MODELS for suffix in (\"_validation_predictions.csv\", \"_error_cases.csv\"))\n\n\ndef seed_manifest(state, cfg):\n    guard_development(state[\"tx\"], cfg)\n    return {\"seed\": state[\"seed\"], \"configuration\": cfg.__dict__, \"timing_checks\": state[\"checks\"],\n            \"model_columns\": state[\"model_columns\"], \"provisional_validation_leader\": state[\"leader\"],\n            \"fit_at\": str(cfg.fit_at), \"validation_at\": str(cfg.validation_at), \"locked_test_at\": str(cfg.test_at),\n            \"last_development_event\": str(state[\"tx\"].timestamp.max()),\n            \"development_dataset_sha256\": hashlib.sha256(pd.util.hash_pandas_object(state[\"tx\"], index=True).values.tobytes()).hexdigest(),\n            \"locked_test_evaluated\": False, \"locked_test_materialized\": False}\n\n\ndef separation_stage(state, cfg, out):\n    tx = state[\"tx\"].loc[state[\"masks\"][\"validation\"]]\n    actual = separation_audit(tx, state[\"features\"], cfg, \"v2_hardened\")\n    actual_overlap = overlap_audit(tx, state[\"features\"], cfg, \"v2_hardened\")\n    _, _, ref_tx = simulate(cfg, state[\"seed\"], hardened=False)\n    ref_f = build_features(ref_tx, cfg)\n    ref_tx = ref_tx.set_index(\"transaction_id\")\n    ref_val = ref_tx.loc[ref_tx.timestamp >= cfg.validation_at]\n    ref = separation_audit(ref_val, ref_f, cfg, \"v1_like_reference_not_archived_v1\")\n    ref_overlap = overlap_audit(ref_val, ref_f, cfg, \"v1_like_reference_not_archived_v1\")\n    audit = pd.concat([ref, actual], ignore_index=True)\n    audit.to_csv(out / \"feature_separation_audit.csv\", index=False)\n    pd.concat([ref_overlap, actual_overlap], ignore_index=True).to_csv(out / \"simulator_overlap_audit.csv\", index=False)\n    table = audit.loc[audit[\"class\"].eq(\"fraud\")].pivot(index=\"feature\", columns=\"world\", values=\"roc_auc_best_direction\")\n    ax = table.reindex(SEPARATION_FEATURES).plot.barh(figsize=(9, 5), xlim=(.45, 1), fontsize=8)\n    ax.set(xlabel=\"Univariate ROC-AUC, better of both directions\", ylabel=\"\", title=\"Development-only simulator separability diagnostic\")\n    ax.legend(fontsize=7)\n    fig = ax.get_figure()\n    fig.tight_layout()\n    fig.savefig(out / \"feature_separation.png\", dpi=150)\n    plt.close(fig)\n\n\ndef robustness_stage(state, cfg, out):\n    per_seed = [state[\"comparison\"].assign(seed=state[\"seed\"])]\n    long_rows, context_rows = [state[\"longitudinal_row\"]], list(state[\"longitudinal_context_rows\"])\n    manifests = [seed_manifest(state, cfg)]\n    for seed in SEEDS[1:]:\n        print(\"Starting declared robustness seed\", seed, flush=True)\n        other = complete_primary(prepare_seed(cfg, seed), cfg)\n        per_seed.append(other[\"comparison\"].assign(seed=seed))\n        row, contexts = longitudinal_challenger(other, cfg, state[\"selected_longitudinal\"])\n        long_rows.append(row)\n        context_rows.extend(contexts)\n        seed_out = out / \"seeds\" / str(seed)\n        save_primary(other, cfg, seed_out)\n        if \"longitudinal_predictions\" in other:\n            other[\"longitudinal_predictions\"].to_csv(seed_out / \"longitudinal_validation_predictions.csv\", index_label=\"transaction_id\")\n        details = seed_manifest(other, cfg)\n        manifests.append(details)\n        (seed_out / \"manifest.json\").write_text(json.dumps(details, indent=2), encoding=\"utf-8\")\n        print(\"Completed declared seed\", seed, \"— reserved final period absent\", flush=True)\n    summary = robustness_summary(per_seed)\n    summary.to_csv(out / \"robustness_summary.csv\", index=False)\n    robustness_chart(summary, out)\n    pd.DataFrame(long_rows).to_csv(out / \"longitudinal_tradeoffs.csv\", index=False)\n    pd.DataFrame(context_rows, columns=[\"seed\", \"context\", \"legitimate_count\", \"baseline_legitimate_reviews\",\n        \"challenger_legitimate_reviews\", \"delta_legitimate_reviews\", \"delta_alert_rate\"]).to_csv(out / \"longitudinal_context_tradeoffs.csv\", index=False)\n    state[\"seed_manifests\"] = manifests\n\n\ndef validate_export(out, cfg):\n    missing = [name for name in REQUIRED_OUTPUTS if not (out/name).is_file()]\n    if missing:\n        raise AssertionError(\"Incomplete V2 output contract: \" + \", \".join(missing))\n    manifest = json.loads((out/\"manifest.json\").read_text(encoding=\"utf-8\"))\n    assert manifest[\"locked_test_evaluated\"] is False\n    assert manifest[\"locked_test_materialized\"] is False\n    assert manifest[\"seed_list\"] == list(SEEDS)\n    for item in manifest[\"seed_manifests\"]:\n        assert item[\"locked_test_evaluated\"] is False and item[\"locked_test_materialized\"] is False\n        assert pd.Timestamp(item[\"last_development_event\"]) < cfg.test_at\n    comparison = pd.read_csv(out/\"comparison.csv\")\n    assert set(comparison.model) == set(MODELS)\n    assert set(comparison.daily_review_cap) == set(cfg.review_budgets)\n    r = pd.read_csv(out/\"robustness_summary.csv\")\n    seeds = r.loc[r.row_type.eq(\"seed\")]\n    assert set(seeds.seed.astype(int)) == set(SEEDS) and len(seeds) == len(SEEDS)*len(MODELS)*len(cfg.review_budgets)\n    assert set(r.row_type) == {\"seed\", \"mean\", \"std\", \"min\", \"max\"}\n    assert np.isfinite(seeds[METRICS].to_numpy()).all()\n    assert len(pd.read_csv(out/\"ablation_summary.csv\")) == len(FEATURE_FAMILIES)+1\n    for path in out.rglob(\"*validation_predictions.csv\"):\n        times = pd.read_csv(path, usecols=[\"timestamp\"])\n        times[\"timestamp\"] = pd.to_datetime(times.timestamp, utc=True)\n        guard_validation(times, cfg)\n    return {\"required_files\": len(REQUIRED_OUTPUTS), \"seed_rows\": len(seeds), \"final_period_absent\": True}\n\n\ndef export_stage(state, cfg, out):\n    save_primary(state, cfg, out)\n    for name, model in state[\"models\"].items():\n        if name.startswith(\"catboost\"):\n            model.save_model(str(out/(name+\".cbm\")))\n        else:\n            joblib.dump(model, out/(name+\".joblib\"))\n    if \"longitudinal_model\" in state:\n        state[\"longitudinal_model\"].save_model(str(out/\"longitudinal_challenger.cbm\"))\n        state[\"longitudinal_predictions\"].to_csv(out/\"longitudinal_validation_predictions.csv\", index_label=\"transaction_id\")\n    pd.DataFrame({\"feature\": FEATURES, \"importance\": state[\"models\"][\"catboost_history\"].get_feature_importance()}).sort_values(\n        \"importance\", ascending=False).to_csv(out/\"feature_importance.csv\", index=False)\n    state[\"tx\"].to_csv(out/\"development_transactions.csv.gz\", index_label=\"transaction_id\", compression=\"gzip\")\n    manifest = {\n        \"project\": \"transaction-fraud-intelligence\", \"implementation_version\": VERSION,\n        \"python_version\": platform.python_version(), \"packages\": {p: importlib.metadata.version(p) for p in CORE_PACKAGES},\n        \"seed_list\": list(SEEDS), \"configuration\": cfg.__dict__, \"feature_list\": FEATURES,\n        \"model_columns\": state[\"model_columns\"], \"categorical_features\": CATEGORICAL_FEATURES,\n        \"diagnostic_features\": DIAGNOSTIC_FEATURES, \"selected_longitudinal_features\": state[\"selected_longitudinal\"],\n        \"longitudinal_model_columns\": state.get(\"longitudinal_columns\", []),\n        \"longitudinal_selection\": {\"seed\": 42, \"period\": \"development_validation\", \"min_finite_per_group\": 5,\n            \"min_coverage_per_group\": .5, \"min_absolute_standardized_difference\": .25,\n            \"separate_from_primary_comparison\": True},\n        \"feature_families\": FEATURE_FAMILIES, \"ablation_seed\": 42,\n        \"split_cutoffs\": {\"fit_at\": str(cfg.fit_at), \"validation_at\": str(cfg.validation_at), \"locked_test_at\": str(cfg.test_at)},\n        \"timing_checks\": state[\"checks\"], \"seed_manifests\": state[\"seed_manifests\"],\n        \"locked_test_evaluated\": False, \"locked_test_materialized\": False,\n        \"selected_provisional_validation_leader\": state[\"leader\"], \"rule_weights\": RULE_WEIGHTS,\n        \"source\": \"synthetic standardized INR payment attempts\", \"score_status\": \"uncalibrated\",\n        \"engine_sha256\": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        \"known_limitations\": [\n            \"Synthetic development results do not establish real-world performance.\",\n            \"The paired V1-like reference is not an exact reproduction of archived V1 data.\",\n            \"Simulator hardening changes the benchmark; V1/V2 score differences are not a pure model comparison.\",\n            \"Labels mature after an assumed seven days for both classes.\",\n            \"Daily top-K is offline ranking of attempts, not online approval, blocking or prevented loss.\",\n            \"Validation selection is exploratory, particularly the primary-seed longitudinal challenger.\",\n            \"Repeated seeds measure synthetic-world variability, not a real-population confidence interval.\",\n            \"Ablations use one declared seed and independent early stopping under equal maximum budgets.\",\n            \"Context tags are exclusive; multiple legitimate changes may overlap in reality.\",\n            \"No final-test events are materialized or examined; later evaluation needs a frozen protocol.\",\n            \"API, deployment, public-data benchmark, SHAP and calibration remain out of scope.\",\n        ],\n    }\n    (out/\"manifest.json\").write_text(json.dumps(manifest, indent=2), encoding=\"utf-8\")\n    check = validate_export(out, cfg)\n    (out/\"output_contract_check.json\").write_text(json.dumps(check, indent=2), encoding=\"utf-8\")\n    (out/\"resolved_packages.txt\").write_text(\"\\n\".join(sorted(\n        d.metadata[\"Name\"]+\"==\"+d.version for d in importlib.metadata.distributions() if d.metadata.get(\"Name\")))+\"\\n\", encoding=\"utf-8\")\n    archive = Path(shutil.make_archive(str(out), \"zip\", root_dir=out))\n    with zipfile.ZipFile(archive) as z:\n        assert set(REQUIRED_OUTPUTS) <= set(z.namelist())\n    return archive\n\n\ndef execute_stage(stage, work, cfg):\n    work = Path(work)\n    work.mkdir(parents=True, exist_ok=True)\n    checkpoint = work/\"state.joblib\"\n    if stage not in STAGES:\n        raise ValueError(\"Unknown experiment stage.\")\n    if stage == \"simulation\":\n        out = work/\"results\"/(\"fraud_v2_\"+datetime.now(timezone.utc).strftime(\"%Y%m%dT%H%M%S_%fZ\"))\n        out.mkdir(parents=True)\n        _, _, tx = simulate(cfg, SEEDS[0])\n        state = {\"seed\": SEEDS[0], \"tx\": tx.set_index(\"transaction_id\"), \"models\": {},\n                 \"model_columns\": {}, \"predictions\": {}, \"completed\": [], \"out\": str(out),\n                 \"configuration\": cfg.__dict__}\n    else:\n        state = joblib.load(checkpoint)\n        if state[\"configuration\"] != cfg.__dict__:\n            raise ValueError(\"Configuration changed mid-run. Begin a fresh run.\")\n        if state[\"completed\"] != list(STAGES[:STAGES.index(stage)]):\n            raise ValueError(\"Run notebook stages in order; repeat Run all for a fresh run.\")\n        out = Path(state[\"out\"])\n    guard_development(state[\"tx\"], cfg)\n    if stage == \"history\":\n        state[\"features\"] = build_features(state[\"tx\"].reset_index(), cfg)\n    elif stage == \"split\":\n        state[\"masks\"], state[\"split_audit\"] = temporal_split(state[\"tx\"], cfg)\n        state[\"checks\"] = timing_checks(state[\"tx\"].reset_index(), state[\"features\"], cfg)\n        state[\"split_audit\"].to_csv(out/\"split_audit.csv\", index=False)\n    elif stage == \"rules\":\n        _, _, state[\"predictions\"][\"rules\"] = score_detector(state, cfg, \"rules\")\n    elif stage in (\"logistic\", \"current\", \"history_model\"):\n        name = {\"logistic\": \"logistic_history\", \"current\": \"catboost_current\", \"history_model\": \"catboost_history\"}[stage]\n        state[\"models\"][name], state[\"model_columns\"][name] = fit_detector(state, cfg, name)\n    elif stage == \"comparison\":\n        complete_primary(state, cfg)\n        save_primary(state, cfg, out)\n        validation_chart(state, cfg, out)\n    elif stage == \"scenario\":\n        scenario_table(state[\"predictions\"]).to_csv(out/\"scenario_breakdown.csv\", index=False)\n        separation_stage(state, cfg, out)\n    elif stage == \"low_slow\":\n        diagnosis, decisions = low_slow_diagnostics(state)\n        diagnosis.to_csv(out/\"low_and_slow_diagnostics.csv\", index=False)\n        diagnosis.pivot(index=\"feature\", columns=\"group\", values=\"median\").reindex([\n            \"amount_ratio_30d\", \"hours_since_previous\", \"count_ratio_7d_prior30\", \"count_ratio_14d_prior30\",\n            \"count_ratio_30d_prior30\", \"attempt_value_7d\", \"new_merchant\", \"category_concentration_7d\",\n            \"prior_accounts_on_device\", \"hour_deviation\"]).reset_index().to_csv(out/\"low_and_slow_overview.csv\", index=False)\n        decisions.to_csv(out/\"longitudinal_feature_decisions.csv\", index=False)\n        state[\"selected_longitudinal\"] = decisions.loc[decisions.selected, \"feature\"].tolist()\n        state[\"longitudinal_row\"], state[\"longitudinal_context_rows\"] = longitudinal_challenger(state, cfg, state[\"selected_longitudinal\"])\n        pd.DataFrame([state[\"longitudinal_row\"]]).to_csv(out/\"longitudinal_tradeoffs.csv\", index=False)\n    elif stage == \"false_positives\":\n        contexts, examples = context_table(state)\n        contexts.to_csv(out/\"legitimate_context_breakdown.csv\", index=False)\n        examples.to_csv(out/\"false_positive_examples.csv\", index=False)\n    elif stage == \"robustness\":\n        robustness_stage(state, cfg, out)\n    elif stage == \"ablation\":\n        run_ablation(state, cfg).to_csv(out/\"ablation_summary.csv\", index=False)\n    elif stage == \"export\":\n        archive = export_stage(state, cfg, out)\n        (work/\"result.json\").write_text(json.dumps({\"output_directory\": str(out), \"archive\": str(archive)}), encoding=\"utf-8\")\n    state[\"completed\"].append(stage)\n    joblib.dump(state, checkpoint)\n    (work/\"progress.json\").write_text(json.dumps({\"output_directory\": str(out), \"completed\": state[\"completed\"]}), encoding=\"utf-8\")\n    print(\"Completed\", stage, \"| final test remains absent and unevaluated\", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--stage\", choices=STAGES)\n    parser.add_argument(\"--all\", action=\"store_true\")\n    parser.add_argument(\"--work-dir\", required=True)\n    parser.add_argument(\"--config\", required=True)\n    args = parser.parse_args()\n    cfg = Config(**json.loads(Path(args.config).read_text(encoding=\"utf-8\")))\n    if args.all or args.stage == \"simulation\":\n        print(\"Python\", platform.python_version(), \"|\", json.dumps({p: importlib.metadata.version(p) for p in CORE_PACKAGES}), flush=True)\n    for stage in STAGES if args.all else [args.stage]:\n        execute_stage(stage, args.work_dir, cfg)\n\n\nif __name__ == \"__main__\":\n    main()\n"
REQUIREMENTS = "numpy==2.2.6\npandas==2.2.3\nscipy==1.15.3\nscikit-learn==1.6.1\ncatboost==1.2.10\nmatplotlib==3.10.1\njoblib==1.4.2\n"

"""Standard-library notebook frontend; scientific packages run in isolation.

The notebook generator supplies ENGINE_SOURCE and REQUIREMENTS before this code.
"""
import csv
import hashlib
import html
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
from datetime import datetime, timezone

from IPython.display import HTML, Image, FileLink, display


def prepare_runtime():
    print("Notebook Python:", platform.python_version(), flush=True)
    if not (3, 11) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError("This release supports Python 3.11–3.13. Choose a supported Colab runtime version.")
    digest = hashlib.sha256(REQUIREMENTS.encode()).hexdigest()[:12]
    runtime = Path.cwd()/".fraud_v2_runtime"/("py%d%d_" % sys.version_info[:2] + digest)
    packages = runtime/"packages"
    marker = runtime/"installed.txt"
    runtime.mkdir(parents=True, exist_ok=True)
    requirements = runtime/"requirements.txt"
    requirements.write_text(REQUIREMENTS, encoding="utf-8")
    if not marker.exists() or marker.read_text() != REQUIREMENTS:
        print("Installing the experiment's isolated packages; the notebook kernel is unchanged.", flush=True)
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
            "--only-binary=:all:", "--ignore-installed", "--upgrade", "--target", str(packages),
            "-r", str(requirements)], capture_output=True, text=True)
        if result.returncode:
            print(result.stdout)
            print(result.stderr)
            raise RuntimeError("Dependency installation failed. The experiment has not started.")
        marker.write_text(REQUIREMENTS, encoding="utf-8")
    env = dict(os.environ, PYTHONPATH=str(packages), PYTHONNOUSERSITE="1", MPLBACKEND="Agg")
    # Import the binaries in a fresh process; do not hide incompatible imports.
    probe = "import numpy,pandas,scipy,sklearn,catboost,matplotlib,joblib; import importlib.metadata as m,json,platform; print('Experiment Python: '+platform.python_version()); print(json.dumps({p:m.version(p) for p in ['numpy','pandas','scipy','scikit-learn','catboost','matplotlib','joblib']},indent=2))"
    subprocess.run([sys.executable, "-S", "-c", probe], env=env, check=True)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    work = Path.cwd()/"outputs"/("v2_work_"+run_id)
    work.mkdir(parents=True)
    engine = work/"fraud_v2.py"
    engine.write_text(ENGINE_SOURCE, encoding="utf-8")
    smoke = os.environ.get("FRAUD_SMOKE_TEST") == "1"
    config = {"customers": 360 if smoke else 1200, "boosting_iterations": 100 if smoke else 600,
              "mode": "smoke" if smoke else "development"}
    config_path = work/"configuration.json"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    print("Five fixed seeds: 42, 123, 2025, 31415, 27182. Mode:", config["mode"], flush=True)
    print("Reserved final period: not generated, inspected or evaluated.", flush=True)
    return work, engine, config_path, env


WORK, ENGINE, CONFIG_PATH, WORKER_ENV = prepare_runtime()


def run_stage(stage):
    command = [sys.executable, "-S", "-u", str(ENGINE), "--stage", stage,
               "--work-dir", str(WORK), "--config", str(CONFIG_PATH)]
    process = subprocess.Popen(command, env=WORKER_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("The '"+stage+"' stage failed. Stop here and share the error output.")


def result_dir():
    return Path(json.loads((WORK/"progress.json").read_text())["output_directory"])


def show_table(filename, columns=None, filters=None, limit=12):
    with (result_dir()/filename).open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        columns = columns or reader.fieldnames
        rows = [r for r in reader if not filters or all(r.get(k) == str(v) for k, v in filters.items())]
    def format_cell(value):
        try:
            return "%.4g" % float(value) if value else ""
        except ValueError:
            return value
    header = "".join("<th>"+html.escape(c)+"</th>" for c in columns)
    body = "".join("<tr>"+"".join("<td>"+html.escape(format_cell(r.get(c, "")))+"</td>" for c in columns)+"</tr>" for r in rows[:limit])
    display(HTML("<table><thead><tr>"+header+"</tr></thead><tbody>"+body+"</tbody></table>"))
    if len(rows) > limit:
        print("Showing", limit, "of", len(rows), "rows. The ZIP contains the full table.")


def show_chart(filename):
    display(Image(filename=str(result_dir()/filename)))


def download_results():
    result = json.loads((WORK/"result.json").read_text())
    archive = Path(result["archive"])
    print("V2 output contract verified. ZIP:", archive)
    if os.environ.get("FRAUD_SMOKE_TEST") == "1":
        display(FileLink(str(archive.relative_to(Path.cwd()))))
        return
    try:
        from google.colab import files
    except ImportError:
        display(FileLink(str(archive.relative_to(Path.cwd()))))
    else:
        files.download(str(archive))

## 3. Simulation

Normal activity includes travel, phone changes, large purchases, legitimate bursts and shared household devices. Fraud sometimes uses a familiar device, familiar merchant, home country, ordinary amount, ordinary hour and no failed-attempt burst.

Account takeover, card testing and low-and-slow remain separate audit scenarios. The simulator uses separate normal/fraud random streams and drops reserved-time candidates before creating outcomes or labels. A paired V1-like reference is used later for diagnostics; it is not a reproduction of the archived V1 dataset.

In [ ]:
run_stage('simulation')

## 4. Point-in-time history builder

The original 21 primary features are preserved. Equal-timestamp attempts are scored together before history updates. Failed outcomes become available only at their release time.

Additional longitudinal measures are **diagnostics first**: 7/14/30-day count and attempted-value changes, ticket-size changes, merchant novelty/diversity and category concentration. Recent windows are compared against a disjoint earlier 30-day baseline, with explicit coverage requirements. Neither the current attempt nor later events enter those baselines.

In [ ]:
run_stage('history')

## 5. Temporal split and label maturity

Fit cutoff: day 72. Early-stopping labels must mature by day 86. Development validation: days 86–101. Both classes have a declared seven-day confirmation delay.

Checks cover future invariance for **all** features, same-timestamp batching, delayed outcomes, exclusion of the current amount, label maturity and the development boundary. Unlabelled observed activity may update history during maturation gaps.

In [ ]:
run_stage('split')
show_table('split_audit.csv')

## 6. Rules baseline

The original five illustrative rules and weights remain fixed. Their output is a 0–100 point score, not a fraud probability.

In [ ]:
run_stage('rules')

## 7. Logistic baseline

The original history feature set is used. Imputation, scaling and category encoding are fitted on training rows only. A convergence failure stops the experiment.

In [ ]:
run_stage('logistic')

## 8. Current-transaction CatBoost

Only log amount, hour sine/cosine, category and country enter this comparator. It is the control for the value of history.

In [ ]:
run_stage('current')

## 9. History CatBoost

The original 21 features enter the primary history model. The same maximum tree budget and early-stopping rules apply across seeds. No parameter sweep is performed.

In [ ]:
run_stage('history_model')

## 10. Main validation comparison

All four detectors use identical validation attempts and 20/50/100 daily review budgets. This is **offline end-of-day top-K**, not live blocking. Ties use transaction ID/event order.

False positives consume limited investigator attention. A fixed budget prevents a model from claiming better recall simply by reviewing everything. Daily summaries include zero-candidate days, exhausted budgets, underfilled days and unused slots. Daily means are unweighted summaries; the main table aggregates all selected attempts.

In [ ]:
run_stage('comparison')
show_table('comparison.csv', columns=['model','daily_review_cap','review_count','fraud_reviewed','average_precision','roc_auc','precision','fraud_recall','false_positive_rate','legitimate_reviewed','fraud_attempt_value_capture'])
show_table('daily_variability_summary.csv', columns=['model','daily_review_cap','validation_days','days_budget_exhausted','days_fewer_candidates_than_budget','unused_review_slots','precision_daily_mean','precision_daily_std','recall_daily_min','recall_daily_max'])
show_chart('validation_comparison.png')

## 11. Scenario analysis and simulator separability

Scenario recall shows which mechanisms remain difficult. The separation audit compares fraud and legitimate distributions and univariate discrimination for the eight strongest V1 signals, including missingness.

The untrained V1-like reference preserves normal-event streams while using the earlier fraud shortcut settings. The overlap audit reports familiar/ordinary conditions, including their intersection. This diagnoses how the task changed; a V1/V2 metric difference cannot be attributed solely to a better model.

In [ ]:
run_stage('scenario')
show_table('scenario_breakdown.csv')
show_table('simulator_overlap_audit.csv', filters={'class':'fraud'}, columns=['world','condition','rows','count','share'], limit=16)
show_chart('feature_separation.png')

## 12. Low-and-slow diagnosis and conditional challenger

Caught low-and-slow attempts, missed attempts and routine legitimate transactions are compared before selecting any extra model features. The ZIP includes distributions for ticket size, inter-attempt timing, 7/14/30-day changes, merchant/category behaviour, sharing, location and hours.

A predeclared gate requires at least five finite observations per comparison group, at least 50% coverage and an absolute standardized mean difference of 0.25. If any candidates pass, a **separate longitudinal challenger** uses that frozen list across all five seeds. It never replaces the primary four comparators.

Seed-42 challenger gains are exploratory because that validation informed selection. The other four worlds provide separate synthetic replication. Trade-off files include low-and-slow and total recall, precision, false positives, and which legitimate contexts absorb extra alerts. If no candidate qualifies, that outcome is exported explicitly.

In [ ]:
run_stage('low_slow')
show_table('low_and_slow_overview.csv', limit=10)
show_table('longitudinal_feature_decisions.csv', columns=['feature','missed_finite_count','routine_finite_count','standardized_mean_difference','selected'], limit=15)
show_table('longitudinal_tradeoffs.csv', columns=['seed','status','baseline_low_and_slow_recall','challenger_low_and_slow_recall','delta_fraud_recall','delta_precision','delta_legitimate_reviewed'])

## 13. False-positive analysis

For the provisional validation leader, inspect routine activity, travel, new phones, large purchases and legitimate bursts. The table reports total/flagged legitimate attempts, review rate, median and upper-quartile scores, and recurring evidence patterns.

Only a few representative false positives are shown. All four models' detailed cases remain in the ZIP. Evidence is an observed feature summary; it is not a causal explanation or proof that the model used each listed signal.

In [ ]:
run_stage('false_positives')
show_table('legitimate_context_breakdown.csv', filters={'provisional_leader':'True'}, columns=['model','context','legitimate_count','legitimate_reviewed','alert_rate','score_median','score_q75','top_evidence_patterns'], limit=5)
show_table('false_positive_examples.csv', columns=['transaction_id','legitimate_context','amount','score','evidence_patterns'], limit=5)

## 14. Multi-seed robustness

Every declared seed runs the same simulator settings, four detectors, feature definitions, time cutoffs and review budgets. A failed seed stops the run rather than disappearing from the report.

`robustness_summary.csv` contains per-seed rows plus mean, sample standard deviation and min/max for each model/budget. The chart reports average precision and rank for every seed. This measures variation between synthetic worlds, not a real-population confidence interval.

In [ ]:
run_stage('robustness')
show_table('robustness_summary.csv', filters={'row_type':'mean','daily_review_cap':50}, columns=['model','average_precision','roc_auc','precision','fraud_recall','fraud_attempt_value_capture','false_positive_rate'], limit=4)
show_chart('robustness_ranking.png')
show_table('longitudinal_tradeoffs.csv', columns=['seed','status','evidence_role','delta_low_and_slow_recall','delta_fraud_recall','delta_precision','delta_legitimate_reviewed'], limit=5)

## 15. Feature-family ablation

On the predeclared primary seed 42, retrain after removing each of six disjoint feature families: amount, velocity/recency, novelty, personal baselines, device network, and circadian timing. Category and country remain as background covariates.

The full model is the reference. Each ablation has the same maximum training budget and time splits, with its own early stopping. Negative metric deltas indicate deterioration for higher-is-better metrics; fewer legitimate reviews is desirable. These are bounded diagnostics, not an extensive search.

In [ ]:
run_stage('ablation')
show_table('ablation_summary.csv', columns=['ablation','average_precision','precision','fraud_recall','false_positive_rate','legitimate_reviewed','fraud_attempt_value_capture','delta_average_precision','delta_fraud_recall'], limit=7)

## 16. Limitations and next decision

- All evidence comes from simulated development data. Hardening can lower scores while improving credibility.
- The main four-model comparison is preserved; the conditional challenger is additional exploratory evidence.
- False-positive costs are represented through review capacity, not invented monetary savings.
- Scores are uncalibrated, evidence is descriptive, and top-K is offline.
- Seed robustness is not proof of generalisation to real clients. Ablations use one seed.
- No reserved final-period events or labels have been materialized, inspected or evaluated.
- APIs, deployed dashboards, public-data benchmarks, SHAP and further infrastructure are outside this pass.

**Next:** return the full V2 ZIP for independent review against the V1 baseline. Do not open the final period or choose a new seed based on these results.

## 17. Export and download the V2 artefacts

The export step validates the required files, the five-seed result grid and development-only prediction timestamps before creating the ZIP. It includes comparisons, daily/scenario/context reports, separability, low-and-slow diagnosis, robustness, ablations, predictions, cases, charts, saved primary models and a detailed manifest.

`locked_test_evaluated` and `locked_test_materialized` must both be **false**. Download the ZIP before closing Colab. The final period remains locked.

In [ ]:
run_stage('export')
download_results()